# PHIVE auf der SNR-9-Simulation – direkter Ground-Truth-Vergleich

Das Modell sieht beim Training ausschließlich die noisy Simulation. Das Clean-FID und die Ground-Truth-Koeffizientenkarten werden nur in diesem Evaluationsnotebook geladen. Verwendet wird immer `last.pt`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path('/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/Denoising')
SIM_ROOT = ROOT.parent / 'MRSISimulator' / 'simulations'
sys.path.insert(0, str(ROOT / 'src')) if str(ROOT / 'src') not in sys.path else None

from denoising.config.build import build_config
from denoising.config.load import load_yaml
from denoising.inference.physics import infer_physics_volume
print('Denoising:', ROOT)
print('Simulation:', SIM_ROOT)

## Einstellungen und Modell laden

In [ ]:
RUN_NAME = 'Sim_N2S_AllReg_0001'
CONFIG_NAME = 'train_physics_7T_phive_Sim.yaml'
CHECKPOINT_NAME = 'last.pt'
Z_SLICE = 23
VOXEL_XY = (32, 32)
GPU_NUMBER = 0

RUN_DIR = ROOT / 'trained_models' / RUN_NAME
CONFIG_PATH = RUN_DIR / CONFIG_NAME
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / CHECKPOINT_NAME
NOISY_PATH = SIM_ROOT / 'Simulation_SNR9' / 'realization_000' / 'simulation_fid_noisy.npy'
MASK_PATH = SIM_ROOT / 'Simulation_SNR9' / 'realization_000' / 'brain_mask.npy'
CLEAN_PATH = SIM_ROOT / 'Simulation_GT' / 'simulation_fid_clean.npy'
GT_MAP_DIR = SIM_ROOT / 'Simulation_GT' / 'coefficient_maps'
for path in (CONFIG_PATH, CHECKPOINT_PATH, NOISY_PATH, MASK_PATH, CLEAN_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

cfg = build_config(load_yaml(CONFIG_PATH))
noisy_fid = np.load(NOISY_PATH, mmap_mode='r')
clean_fid = np.load(CLEAN_PATH, mmap_mode='r')
brain_mask = np.load(MASK_PATH).astype(bool)
if noisy_fid.shape != clean_fid.shape or noisy_fid.shape[:3] != brain_mask.shape:
    raise ValueError((noisy_fid.shape, clean_fid.shape, brain_mask.shape))
if not 0 <= Z_SLICE < noisy_fid.shape[2]:
    raise IndexError(Z_SLICE)

mask_slice = brain_mask[:, :, Z_SLICE]
DEVICE = torch.device(f'cuda:{GPU_NUMBER}' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Noisy FID:', noisy_fid.shape, noisy_fid.dtype)
print('Training normalization:', bool(cfg.data.normalization))

## Vollständiges Volumen mit der zentralen Physics-Inference inferieren

In [ ]:
result = infer_physics_volume(
    cfg=cfg,
    checkpoint_path=CHECKPOINT_PATH,
    input_fid=NOISY_PATH,
    device=DEVICE,
    slice_batch_size=1,
)
noisy_spectra = np.fft.fftshift(np.fft.fft(noisy_fid[:, :, Z_SLICE], axis=-1), axes=-1)
clean_spectra = np.fft.fftshift(np.fft.fft(clean_fid[:, :, Z_SLICE], axis=-1), axes=-1)
reconstruction = result.reconstruction_spectrum[:, :, Z_SLICE]
amplitude_maps = result.parameters.amplitudes[:, :, Z_SLICE]
basis_names = result.basis_names
print('Reconstruction:', reconstruction.shape)
print('Amplitude maps:', amplitude_maps.shape, basis_names)
print('Epoch:', result.metadata['checkpoint_epoch'], 'val:', result.metadata['checkpoint_val_loss'])
print('Global input scale restored:', result.normalization_scale)

## Noisy Input, PHIVE und Clean-GT an einem Voxel

In [ ]:
voxel_x, voxel_y = VOXEL_XY
if not mask_slice[voxel_x, voxel_y]:
    coordinates = np.argwhere(mask_slice)
    center = np.asarray(mask_slice.shape) / 2
    voxel_x, voxel_y = coordinates[np.argmin(np.sum((coordinates - center) ** 2, axis=1))]
frequency_hz = result.frequency_axis_hz
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for ax, component, title in ((axes[0], np.real, 'Real'), (axes[1], np.imag, 'Imaginary')):
    ax.plot(frequency_hz, component(noisy_spectra[voxel_x, voxel_y]), alpha=.55, label='Noisy SNR 9')
    ax.plot(frequency_hz, component(reconstruction[voxel_x, voxel_y]), lw=1.5, label='PHIVE')
    ax.plot(frequency_hz, component(clean_spectra[voxel_x, voxel_y]), '--', lw=1.3, label='Clean GT')
    ax.set_title(title); ax.set_xlabel('Frequency offset [Hz]'); ax.grid(alpha=.2); ax.legend()
fig.suptitle(f'z={Z_SLICE}, voxel=({voxel_x}, {voxel_y}), epoch={result.metadata["checkpoint_epoch"]}')
plt.show()

## Letzte Zelle: PHIVE-Parameterkarten gegen Ground Truth

GT und PHIVE verwenden dieselbe Basis und globale FID-Skala. Die Inference-Funktion hat die intern verwendete Normalisierung bereits rückgängig gemacht. Deshalb erhalten beide pro Metabolit bewusst dieselbe Farbskala. Die Fehlerkarte besitzt eine symmetrische eigene Skala.

In [ ]:
gt_maps = {}
pred_maps = {}

for index, name in enumerate(basis_names):
    path = GT_MAP_DIR / f'{name}.npy'
    if not path.is_file():
        raise FileNotFoundError(path)

    volume = np.load(path, mmap_mode='r')
    if volume.shape != brain_mask.shape:
        raise ValueError(f'{name}: GT shape {volume.shape} != {brain_mask.shape}')

    gt_maps[name] = np.asarray(volume[:, :, Z_SLICE])
    pred_maps[name] = amplitude_maps[..., index]

sum_groups = {
    'tNAA': ('NAA', 'NAAG'),
    'tCr': ('Cr', 'PCr'),
    'Glx': ('Glu', 'Gln'),
}

display_names = {
    'tNAA': 'tNAA (NAA + NAAG)',
    'tCr': 'tCr (Cr + PCr)',
    'Glx': 'Glx (Glu + Gln)',
}

for total, components in sum_groups.items():
    gt_maps[total] = sum(gt_maps[name] for name in components)
    pred_maps[total] = sum(pred_maps[name] for name in components)

plot_names = list(basis_names) + list(sum_groups)

n_rows = len(plot_names)
fig, axes = plt.subplots(
    n_rows,
    3,
    figsize=(12, 3.4 * n_rows),
    constrained_layout=True,
    squeeze=False,
)

metrics = []

for row, name in enumerate(plot_names):
    gt = gt_maps[name]
    predicted = pred_maps[name]
    label = display_names.get(name, name)

    valid = mask_slice & np.isfinite(gt) & np.isfinite(predicted)
    error = predicted - gt

    combined = np.concatenate((gt[valid], predicted[valid]))
    vmin, vmax = np.percentile(combined, [1, 99])
    if vmax <= vmin:
        vmax = vmin + 1e-8

    error_limit = np.percentile(np.abs(error[valid]), 99)
    if error_limit <= 0:
        error_limit = 1e-8

    rmse = float(np.sqrt(np.mean(error[valid] ** 2)))
    gt_rms = float(np.sqrt(np.mean(gt[valid] ** 2)))
    nrmse = rmse / gt_rms if gt_rms > 0 else np.nan

    correlation = (
        float(np.corrcoef(gt[valid], predicted[valid])[0, 1])
        if np.std(gt[valid]) > 0 and np.std(predicted[valid]) > 0
        else np.nan
    )

    metrics.append((label, rmse, nrmse, correlation))

    for column, values, title, cmap, lo, hi in (
        (0, gt, 'Ground truth', 'magma', vmin, vmax),
        (1, predicted, 'PHIVE', 'magma', vmin, vmax),
        (2, error, 'PHIVE − GT', 'coolwarm', -error_limit, error_limit),
    ):
        image = axes[row, column].imshow(
            np.where(mask_slice, values, np.nan).T,
            origin='lower',
            cmap=cmap,
            vmin=lo,
            vmax=hi,
        )

        axes[row, column].set_title(f'{label} — {title}')
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
        fig.colorbar(image, ax=axes[row, column], fraction=0.046, pad=0.03)

fig.suptitle(
    f'PHIVE SNR 9 versus exact coefficient GT — '
    f'z={Z_SLICE}, epoch={result.metadata["checkpoint_epoch"]}',
    fontsize=16,
)

plt.show()

print('Metabolite                   RMSE       NRMSE       correlation')
for name, rmse, nrmse, correlation in metrics:
    print(f'{name:26s}  {rmse:10.4g}  {nrmse:10.4f}  {correlation:12.4f}')